# April 05
我把utils.py里面的Reward Signals给重新整理了下然后现在有6种不同的signals.
直接用train_ray_selfplay.py 去从头训练这个新的Reward shape

Submit 了2个Train 一个CPU版本一个GPU版本
等看看训练结果，如果GPU没问题以后就只用GPU训练就好了。

NEXT:
出结果后和ceia_baseline_agent 打个比赛看看效果。

In [ ]:
#SBATCH --job-name=GPU_S2_selfplay
#SBATCH -N1 --ntasks-per-node=12 --gres=gpu:V100:1
#SBATCH --mem-per-cpu=6G
#SBATCH -t12:00:00
#SBATCH --output=soccerstwos-%j.out
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=frank.yang@gatech.edu


source ~/miniconda3/etc/profile.d/conda.sh
conda activate soccertwos

# Fix protobuf version conflict with ray==1.4.0
export PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
pip install "numpy==1.23.5" -q  # fix numpy version conflict with ray==1.4.0

# Run the training script
/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/bin/python train_ray_selfplay.py


# April 06
GPU在这个项目里只是把 PPO 的模型更新（forward/backward pass）放到 GPU 上，但 soccer-twos 的环境是 Unity 进程，仿真本身无法用 GPU 加速，而且 MLP 太小，GPU 收益可以忽略不计。
之后训练直接用CPU的就行。


### 当前脚本不支持多节点
train_ray_selfplay.py:40
ray.init()  # 只初始化本地 Ray，不知道其他节点
多节点需要改成 ray.init(address='auto') 并在 Slurm 脚本里手动启动 Ray 集群（head + workers），比较复杂。

### 修复不能继续训练的问题
问题：checkpoint 序列化时的类型信息丢失。
具体链路是这样的：
Ray 保存 checkpoint 时，把 optimizer state dict（里面是 torch tensor）先转成 numpy array，再用 pickle 序列化存盘
某些 tensor（比如 Adam 的 step 计数器，是一个标量 int64）在转换过程中被存成了 numpy.object_ 类型，而不是具体的数值类型
恢复时 Ray 想把这些 numpy array 转回 torch tensor，调用 torch.from_numpy()，但 torch 拒绝转换 object_ 类型

修复方式：
去到/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/lib/python3.8/site-packages/ray/rllib/utils/torch_utils.py
把def mapping(item)里面的
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    tensor = torch.from_numpy(np.asarray(item))
```
替换为：
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    arr = np.asarray(item)
    if arr.dtype == object:
        arr = np.array(arr.tolist(), dtype=np.float32)
    # Scalar values (like lr) should stay as Python scalars, not tensors
    if arr.ndim == 0:
        return arr.item()
    tensor = torch.from_numpy(arr)
```


# April 07
用Ray 1.4版本重新跑

# April 08
用`python -m soccer_twos.evaluate     -m1 reward_shaping_ppo_agent     -m2 ceia_baseline_agent     -e 100`
比较了下 Ray1.4 的reward shaping 版本和 ceia 发现：
reward shaping 版本比 TA 的原始 ceia baseline 强 9 个百分点。
需要继续训练

# April 09
目前有几个训练的结果
- `PPO_Soccer_4923f_00000_0_2026-04-08_21-56-12` 这个是Ray1.13 的1300版本
- `PPO_Soccer_36ca0_00000_0_2026-04-07_18-00-41` 这个是Ray1.4 的1000 版本
- `PPO_Soccer_1874b_00000_0_2026-04-09_13-18-16` Ray1.4版本 所有的环境记录在 environment.yml 里面



# April 10
`PPO_Soccer_bc696_00000_0_2026-04-09_14-12-58` 
- Ray1.4 并且消除了 `DeprecationWarning: wrapping <function policy_mapping_fn at `  
- 训练出`scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_bc696_00000_0_2026-04-09_14-12-58/checkpoint_000700/checkpoint-700`
- 从这儿继续训练 `PPO_Soccer_0b340_00000_0_2026-04-10_15-47-02`

### 对比测试：
`python -m soccer_twos.watch -m1 reward_shaping_ppo_agent -m2 ceia_baseline_agent`
- 上面这个需要显示出来只能本地
- 服务器上用这个：
```
python -m soccer_twos.evaluate \
    -m1 reward_shaping_ppo_agent \
    -m2 ceia_baseline_agent \
    -e 100
```

### checkpoint-700 拉完了
```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 79.32
episode_reward_max: -0.013599991798400879
episode_reward_mean: -0.15827199816703796
episode_reward_min: -1.6103999614715576
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 19
      policy_blue_team_reward_max: 1.981600046157837
      policy_blue_team_reward_mean: 0.39344000816345215
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.62
      policy_blue_team_wins: 31
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 27
      policy_orange_team_reward_max: 1.9847999811172485
      policy_orange_team_reward_mean: -0.2342880219221115
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.46
      policy_orange_team_wins: 23
    policy_draws: 0
    policy_losses: 46
    policy_reward_max: 1.9847999811172485
    policy_reward_mean: 0.07957600802183151
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.54
    policy_wins: 54
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 23
      policy_blue_team_reward_max: 1.979599952697754
      policy_blue_team_reward_mean: 0.06441599130630493
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.54
      policy_blue_team_wins: 27
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 31
      policy_orange_team_reward_max: 1.9864000082015991
      policy_orange_team_reward_mean: -0.5401120185852051
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.38
      policy_orange_team_wins: 19
    policy_draws: 0
    policy_losses: 54
    policy_reward_max: 1.9864000082015991
    policy_reward_mean: -0.23784799873828888
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.46
    policy_wins: 46
```

# April 11
拿到`scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_0b340_00000_0_2026-04-10_15-47-02/checkpoint_002000/checkpoint-2000`
和ceia_baseline_agent 比赛结果极其不理想。

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 73.18
episode_reward_max: -0.01639997959136963
episode_reward_mean: -0.1459999978542328
episode_reward_min: -0.6967999935150146
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 5
      policy_blue_team_reward_max: 1.9836000204086304
      policy_blue_team_reward_mean: 1.452552080154419
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.9
      policy_blue_team_wins: 45
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 10
      policy_orange_team_reward_max: 1.9836000204086304
      policy_orange_team_reward_mean: 1.0916880369186401
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.8
      policy_orange_team_wins: 40
    policy_draws: 0
    policy_losses: 15
    policy_reward_max: 1.9836000204086304
    policy_reward_mean: 1.2721199989318848
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.85
    policy_wins: 85
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 40
      policy_blue_team_reward_max: 1.9648000001907349
      policy_blue_team_reward_mean: -1.230159878730774
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.2
      policy_blue_team_wins: 10
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 45
      policy_orange_team_reward_max: 1.9783999919891357
      policy_orange_team_reward_mean: -1.6060800552368164
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.1
      policy_orange_team_wins: 5
    policy_draws: 0
    policy_losses: 85
    policy_reward_max: 1.9783999919891357
    policy_reward_mean: -1.4181201457977295
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.15
    policy_wins: 15
```

### 核心 Bug：对手永远不会更新
看 train_ray_selfplay.py:27：


class SelfPlayUpdateCallback(DefaultCallbacks):
    def on_train_result(self, **info):
        if info["result"]["episode_reward_mean"] > 0.5:  # ← 问题在这
            print("---- Updating opponents!!! ----")
这个条件 永远不会 True。

episode_reward_mean 是含 shaped reward 的总奖励，来自 utils.py:35：


combined = {aid: reward[aid] + shaped[aid] for aid in reward}
而 shaped reward 里有两个常驻负信号：

Signal 5（每步时间惩罚）：-0.001 × 步数，episode 平均 ~75 步 → 每 episode -0.075
Signal 4（危险防守）：只要球在场上，随时在扣分
你的实际数据验证了这一点：

April 10: episode_reward_mean = -0.158
April 11: episode_reward_mean = -0.146
始终是负数，从未超过 0.5。

后果

opponent_1 / opponent_2 / opponent_3 在整个训练过程中从未更新
↓
它们永远冻结在 checkpoint-700 的权重
↓
default policy 花 1300 个 iteration 专门对付 checkpoint-700 版本的自己
↓
学会了针对特定模式的"取巧策略"，对 ceia_baseline_agent 完全无效
这就解释了为什么越训练越烂：不是 agent 变笨了，而是它过拟合了一个永远不会进化的对手。

次要问题
[utils.py:27]：episode_reward_mean 是 0.5 的阈值是对 raw reward 校准的（原始 soccer-twos 信号是 ±2），加了 shaped reward 后这个阈值就完全错了。


### NEXT：
我修改成`if info["result"]["episode_reward_mean"] > -0.05:`
然后从checkpoint-700 继续训练 
- `PPO_Soccer_750bc_00000_0_2026-04-11_14-30-04`

我修改了训练脚本成：
`"num_workers": 7,
"num_envs_per_worker": 3,`
这样Ray 进程间通信开销减少（22 个进程 → 7 个进程）
每个 worker 的 rollout 收集更连续，batch 质量更好
并行环境数量差不多（22 → 21），吞吐量基本持平
- `PPO_Soccer_6cc90_00000_0_2026-04-11_15-27-06`







# April 12

`scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_750bc_00000_0_2026-04-11_14-30-04/checkpoint_001900/checkpoint-1900` 和baseline 比赛结果：

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 55.85
episode_reward_max: -0.014799952507019043
episode_reward_mean: -0.11124400049448013
episode_reward_min: -0.45280003547668457
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 9
      policy_blue_team_reward_max: 1.9747999906539917
      policy_blue_team_reward_mean: 1.1871360540390015
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.82
      policy_blue_team_wins: 41
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 11
      policy_orange_team_reward_max: 1.983199954032898
      policy_orange_team_reward_mean: 1.023919939994812
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.78
      policy_orange_team_wins: 39
    policy_draws: 0
    policy_losses: 20
    policy_reward_max: 1.983199954032898
    policy_reward_mean: 1.1055281162261963
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.8
    policy_wins: 80
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 39
      policy_blue_team_reward_max: 1.9811999797821045
      policy_blue_team_reward_mean: -1.1410319805145264
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.22
      policy_blue_team_wins: 11
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 41
      policy_orange_team_reward_max: 1.985200047492981
      policy_orange_team_reward_mean: -1.2925119400024414
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.18
      policy_orange_team_wins: 9
    policy_draws: 0
    policy_losses: 80
    policy_reward_max: 1.985200047492981
    policy_reward_mean: -1.2167720794677734
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.2
    policy_wins: 20
```

```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 75.78
episode_reward_max: -0.018800020217895508
episode_reward_mean: -0.15107598900794983
episode_reward_min: -0.809999942779541
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 8
      policy_blue_team_reward_max: 1.979200005531311
      policy_blue_team_reward_mean: 1.2336320877075195
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.84
      policy_blue_team_wins: 42
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 7
      policy_orange_team_reward_max: 1.9811999797821045
      policy_orange_team_reward_mean: 1.321287989616394
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.86
      policy_orange_team_wins: 43
    policy_draws: 0
    policy_losses: 15
    policy_reward_max: 1.9811999797821045
    policy_reward_mean: 1.277459979057312
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.85
    policy_wins: 85
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 43
      policy_blue_team_reward_max: 1.9775999784469604
      policy_blue_team_reward_mean: -1.456527829170227
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.14
      policy_blue_team_wins: 7
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 42
      policy_orange_team_reward_max: 1.9579999446868896
      policy_orange_team_reward_mean: -1.4005439281463623
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.16
      policy_orange_team_wins: 8
    policy_draws: 0
    policy_losses: 85
    policy_reward_max: 1.9775999784469604
    policy_reward_mean: -1.428536057472229
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.15
    policy_wins: 15
```

#### 问题：
两个奖励差了 100 倍。 训练时 agent 优化的是 shaped reward（15.15），评估时计分用的是 raw game reward（±2 max）。Agent 学会了刷 shaped reward，没有学会赢球。

看下两个 shaped reward 的量级对比：

信号	系数	每步估计量	60步 episode 合计
接近球 Signal 1	0.01	~0.03	~1.8/agent
进攻方向 Signal 3	0.02	~0.1	~6/agent
踢球 Signal 2	0.05	偶发	~0.5/agent
4个 agent 总计			~30+
真实游戏信号		进球才有	±1/goal
shaped reward 是 game signal 的 30-60 倍，agent 根本无法学到"赢球"这个目标。

#### 处理：

改动	文件	效果
所有 shaped reward 系数 ÷50	utils.py	game signal 重新主导，training/eval 一致
callback 阈值改回 > 0.3	train_ray_selfplay.py	对应 raw reward 量级，"赢了才更新对手"
去掉 restore	train_ray_selfplay.py	从头训练，避免带着错误习惯继续
从头训练后，训练里的 episode_reward_mean 应该在 -0.5 到 +1.5 之间（接近 raw game reward 量级），这才是正确的。